# LanceDB fundamentals

In [1]:
import lancedb 
from constants import KNOWLEDGE_BASE_PATH

vector_db = lancedb.connect(KNOWLEDGE_BASE_PATH)
vector_db

LanceDBConnection(uri='/Users/aigineer/Documents/github/llmops_kokchun_giang_mlo25/09_lancedb/knowledge_base')

In [2]:
import json

with open("animals_text_embeddings.json") as file:
    data = json.load(file)

data

[{'text': 'A small brown dog running.', 'vector': [0.12, 0.85, 0.33]},
 {'text': 'A cat resting quietly on a sofa.', 'vector': [0.4, 0.91, 0.1]},
 {'text': 'A large gray elephant drinking water.',
  'vector': [0.88, 0.22, 0.55]},
 {'text': 'A fast cheetah sprinting across the savannah.',
  'vector': [0.95, 0.12, 0.72]},
 {'text': 'A colorful parrot perched on a branch.',
  'vector': [0.25, 0.66, 0.81]},
 {'text': 'A frog sitting on a lily pad.', 'vector': [0.14, 0.44, 0.27]}]

In [3]:
vector_db.create_table("animals_text", exist_ok=True, data=data)
vector_db["animals_text"]

LanceTable(name='animals_text', version=1, _conn=LanceDBConnection(uri='/Users/aigineer/Documents/github/llmops_kokchun_giang_mlo25/09_lancedb/knowledge_base'))

In [5]:
vector_db["animals_text"].to_pandas()

,text,vector
0,A small brown dog running.,"[0.12, 0.85, 0.33]"
1,A cat resting quietly on a sofa.,"[0.4, 0.91, 0.1]"
2,A large gray elephant drinking water.,"[0.88, 0.22, 0.55]"
3,A fast cheetah sprinting across the savannah.,"[0.95, 0.12, 0.72]"
4,A colorful parrot perched on a branch.,"[0.25, 0.66, 0.81]"
5,A frog sitting on a lily pad.,"[0.14, 0.44, 0.27]"


In [6]:
vector_db["animals_text"].head()

pyarrow.Table
text: string
vector: fixed_size_list<item: float>[3]
  child 0, item: float
----
text: [["A small brown dog running.","A cat resting quietly on a sofa.","A large gray elephant drinking water.","A fast cheetah sprinting across the savannah.","A colorful parrot perched on a branch."]]
vector: [[[0.12,0.85,0.33],[0.4,0.91,0.1],[0.88,0.22,0.55],[0.95,0.12,0.72],[0.25,0.66,0.81]]]

## Add more data

In [8]:
more_data = [
    {"text": "A panda eating bamboo peacefully.", "vector": [0.51, 0.37, 0.82]},
    {"text": "A lion roaring loudly on a rock.", "vector": [0.93, 0.18, 0.41]},
]

vector_db["animals_text"].add(more_data)

AddResult(version=2)

In [9]:
vector_db["animals_text"].to_pandas()

,text,vector
0,A small brown dog running.,"[0.12, 0.85, 0.33]"
1,A cat resting quietly on a sofa.,"[0.4, 0.91, 0.1]"
2,A large gray elephant drinking water.,"[0.88, 0.22, 0.55]"
3,A fast cheetah sprinting across the savannah.,"[0.95, 0.12, 0.72]"
4,A colorful parrot perched on a branch.,"[0.25, 0.66, 0.81]"
5,A frog sitting on a lily pad.,"[0.14, 0.44, 0.27]"
6,A panda eating bamboo peacefully.,"[0.51, 0.37, 0.82]"
7,A lion roaring loudly on a rock.,"[0.93, 0.18, 0.41]"


## Create empty table

In [12]:
from lancedb.pydantic import LanceModel

class Article(LanceModel):
    title: str 
    content: str 

vector_db.create_table("articles", schema=Article, exist_ok=True)


LanceTable(name='articles', version=1, _conn=LanceDBConnection(uri='/Users/aigineer/Documents/github/llmops_kokchun_giang_mlo25/09_lancedb/knowledge_base'))

In [14]:
vector_db.list_tables()

ListTablesResponse(tables=['animals_text', 'articles'], page_token=None)

In [16]:
articles_data = [
    {
        "title": "Fiskar",
        "content": """Fiskar (Pisces) är en grupp vattenlevande ryggradsdjur med fenor, som indelas i benfiskar, broskfiskar och käklösa fiskar. De flesta arter andas med gälar och är växelvarma. Undantaget är lungfiskar. Eftersom landryggradsdjuren släktskapsmässigt är en typ av kvastfeniga fiskar men inte klassificeras som fiskar är begreppet "fiskar" parafyletiskt. Vetenskapen om fiskar kallas iktyologi.""",
    },
    {
        "title": "Krabba",
        "content": """Krabbor (Brachyura) är en delordning av vattenlevande kräftdjur. Vissa arter fiskas som mat, ofta ansedda som delikatesser. Krabbor har tio ben varav det första paret bär gripklor.""",
    },
]

vector_db["articles"].add(articles_data)

AddResult(version=2)

In [17]:
vector_db["articles"].to_pandas()

,title,content
0,Fiskar,Fiskar (Pisces) är en grupp vattenlevande rygg...
1,Krabba,Krabbor (Brachyura) är en delordning av vatten...


## Drop a table

In [19]:
vector_db.drop_table("articles")

In [ ]:
vector_db.list_tables() # .table_names() for intel macs

ListTablesResponse(tables=['animals_text'], page_token=None)

## Vector search in LanceDB

Flow here (common approach in many vector databases): 
1. use embedding model to create vector embeddings for each document
2. use same embedding model to create vector embedding of the query
3. send in the query_vector to search for closest documents
   
If we want to do RAG -> send in closest document to LLM to use as context

In [23]:
vector_db["animals_text"].to_pandas()

,text,vector
0,A small brown dog running.,"[0.12, 0.85, 0.33]"
1,A cat resting quietly on a sofa.,"[0.4, 0.91, 0.1]"
2,A large gray elephant drinking water.,"[0.88, 0.22, 0.55]"
3,A fast cheetah sprinting across the savannah.,"[0.95, 0.12, 0.72]"
4,A colorful parrot perched on a branch.,"[0.25, 0.66, 0.81]"
5,A frog sitting on a lily pad.,"[0.14, 0.44, 0.27]"
6,A panda eating bamboo peacefully.,"[0.51, 0.37, 0.82]"
7,A lion roaring loudly on a rock.,"[0.93, 0.18, 0.41]"


In [29]:
query_vector = [0.5, 0.2, 0.9]
df = vector_db["animals_text"].search(query_vector).limit(3).to_pandas()
df

,text,vector,_distance
0,A panda eating bamboo peacefully.,"[0.51, 0.37, 0.82]",0.0354
1,A fast cheetah sprinting across the savannah.,"[0.95, 0.12, 0.72]",0.2413
2,A large gray elephant drinking water.,"[0.88, 0.22, 0.55]",0.2673


## Embeddings API in LanceDB

- makes life simpler
- automatically generate embeddings

embedding models can be found here for cohere
- [embedding models](https://docs.cohere.com/docs/cohere-embed)

In [40]:
from lancedb.embeddings import get_registry

embedding_model = get_registry().get("cohere").create(name = "embed-multilingual-v3.0")

embedding_model.ndims()

1024

In [43]:
from lancedb.pydantic import Vector

class JokeModel(LanceModel):
    joke: str = embedding_model.SourceField()
    embedding: Vector(embedding_model.ndims()) = embedding_model.VectorField() # type:ignore

vector_db.create_table("jokes", schema=JokeModel, exist_ok=True)
vector_db["jokes"]



LanceTable(name='jokes', version=1, _conn=LanceDBConnection(uri='/Users/aigineer/Documents/github/llmops_kokchun_giang_mlo25/09_lancedb/knowledge_base'))